# Project 97 — Local API Spec Explainer
## Explain API Schema and Generate Examples

**Stack:** Ollama · LangChain · Structured Parsing · Jupyter

## Project Overview

This notebook builds a **local API spec explainer** that takes an OpenAPI
specification, parses it into structured endpoint data, and uses a local LLM
to generate plain-English documentation, code examples in multiple languages,
and a quickstart guide.

Everything runs **locally** via Ollama. No API specs leave your machine.

### What You Will Learn

1. How to **parse OpenAPI specs** into structured endpoint data
2. How to generate **plain-English explanations** for each endpoint
3. How to generate **code examples** in Python, JavaScript, and cURL
4. How to produce a **quickstart guide** from API metadata
5. How to verify **documentation completeness** against the spec
6. Practical prompt-engineering patterns for API documentation

## Prerequisites

| Requirement | Details |
|---|---|
| **Ollama** | Running locally with `qwen3:8b` pulled |
| **Python packages** | `langchain`, `langchain-ollama` |

```bash
ollama pull qwen3:8b
```

In [ ]:
# !pip install -q langchain langchain-ollama

## Step 1 — Imports and Configuration

In [ ]:
import json

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

OLLAMA_MODEL = "qwen3:8b"
TEMPERATURE = 0.2

print("Configuration ready.")

## Step 2 — Initialize LLM

In [ ]:
llm = ChatOllama(model=OLLAMA_MODEL, temperature=TEMPERATURE)

test_msg = llm.invoke("Reply with only: OK")
print(f"LLM ready: {test_msg.content.strip()[:20]}")

## Step 3 — Define Sample OpenAPI Specification

We define a realistic OpenAPI 3.0 spec for a **Task Manager API** with
five endpoints covering CRUD operations, query parameters, request bodies,
and multiple response codes.

In [ ]:
API_SPEC = {
    "openapi": "3.0.0",
    "info": {
        "title": "Task Manager API",
        "version": "2.0",
        "description": "A REST API for managing tasks with priorities, due dates, and status tracking.",
    },
    "servers": [{"url": "https://api.taskmanager.io/v2"}],
    "paths": {
        "/tasks": {
            "get": {
                "summary": "List tasks",
                "description": "Retrieve a paginated list of tasks, optionally filtered by status.",
                "parameters": [
                    {"name": "status", "in": "query", "required": False,
                     "schema": {"type": "string", "enum": ["pending", "done", "archived"]}},
                    {"name": "limit", "in": "query", "required": False,
                     "schema": {"type": "integer", "default": 20, "maximum": 100}},
                    {"name": "offset", "in": "query", "required": False,
                     "schema": {"type": "integer", "default": 0}},
                ],
                "responses": {
                    "200": {"description": "Array of task objects"},
                    "401": {"description": "Unauthorized — invalid or missing API key"},
                },
            },
            "post": {
                "summary": "Create task",
                "description": "Create a new task. Title is required.",
                "requestBody": {
                    "required": True,
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "required": ["title"],
                                "properties": {
                                    "title": {"type": "string", "maxLength": 200},
                                    "description": {"type": "string"},
                                    "priority": {"type": "string", "enum": ["low", "medium", "high"]},
                                    "due_date": {"type": "string", "format": "date"},
                                },
                            }
                        }
                    },
                },
                "responses": {
                    "201": {"description": "Task created successfully"},
                    "400": {"description": "Validation error — check request body"},
                    "401": {"description": "Unauthorized"},
                },
            },
        },
        "/tasks/{id}": {
            "get": {
                "summary": "Get task by ID",
                "description": "Retrieve a single task by its unique ID.",
                "parameters": [
                    {"name": "id", "in": "path", "required": True,
                     "schema": {"type": "integer"}},
                ],
                "responses": {
                    "200": {"description": "Task object"},
                    "404": {"description": "Task not found"},
                },
            },
            "put": {
                "summary": "Update task",
                "description": "Update an existing task. Send only fields to change.",
                "parameters": [
                    {"name": "id", "in": "path", "required": True,
                     "schema": {"type": "integer"}},
                ],
                "requestBody": {
                    "content": {
                        "application/json": {
                            "schema": {
                                "type": "object",
                                "properties": {
                                    "title": {"type": "string"},
                                    "status": {"type": "string", "enum": ["pending", "done", "archived"]},
                                    "priority": {"type": "string", "enum": ["low", "medium", "high"]},
                                },
                            }
                        }
                    },
                },
                "responses": {
                    "200": {"description": "Task updated"},
                    "404": {"description": "Task not found"},
                },
            },
            "delete": {
                "summary": "Delete task",
                "description": "Permanently delete a task.",
                "parameters": [
                    {"name": "id", "in": "path", "required": True,
                     "schema": {"type": "integer"}},
                ],
                "responses": {
                    "204": {"description": "Task deleted"},
                    "404": {"description": "Task not found"},
                },
            },
        },
    },
}

print(f"API: {API_SPEC['info']['title']} v{API_SPEC['info']['version']}")
print(f"Base URL: {API_SPEC['servers'][0]['url']}")
total_endpoints = sum(len(methods) for methods in API_SPEC['paths'].values())
print(f"Endpoints: {total_endpoints}")

## Step 4 — Parse API Spec into Structured Endpoint Data

We extract each endpoint into a flat structure with method, path,
parameters, request body, and responses — making it easier to
process individually.

In [ ]:
def parse_openapi_spec(spec: dict) -> list[dict]:
    """Parse an OpenAPI spec into a flat list of endpoint records."""
    endpoints = []
    base_url = spec.get("servers", [{}])[0].get("url", "")

    for path, methods in spec.get("paths", {}).items():
        for method, details in methods.items():
            # Extract parameters
            params = []
            for p in details.get("parameters", []):
                params.append({
                    "name": p["name"],
                    "location": p["in"],
                    "required": p.get("required", False),
                    "type": p.get("schema", {}).get("type", "string"),
                    "enum": p.get("schema", {}).get("enum"),
                    "default": p.get("schema", {}).get("default"),
                })

            # Extract request body schema
            body_schema = None
            rb = details.get("requestBody", {})
            if rb:
                content = rb.get("content", {})
                json_schema = content.get("application/json", {}).get("schema")
                if json_schema:
                    body_schema = json_schema

            # Extract responses
            responses = []
            for code, resp in details.get("responses", {}).items():
                responses.append({"code": code, "description": resp.get("description", "")})

            endpoints.append({
                "method": method.upper(),
                "path": path,
                "full_url": base_url + path,
                "summary": details.get("summary", ""),
                "description": details.get("description", ""),
                "parameters": params,
                "request_body": body_schema,
                "responses": responses,
            })

    return endpoints


ENDPOINTS = parse_openapi_spec(API_SPEC)

print(f"Parsed {len(ENDPOINTS)} endpoints:\n")
for ep in ENDPOINTS:
    params_str = ", ".join(p["name"] for p in ep["parameters"])
    body_str = "+ body" if ep["request_body"] else ""
    print(f"  {ep['method']:<7} {ep['path']:<20} {ep['summary']:<20} params=[{params_str}] {body_str}")

## Step 5 — Generate Plain-English Endpoint Explanations

For each endpoint, we ask the LLM to produce a **beginner-friendly
explanation** that a non-technical person could understand.

In [ ]:
EXPLAIN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Explain this API endpoint in plain English for a developer
who has never used this API before.

Include:
1. WHAT IT DOES: One sentence summary
2. WHEN TO USE IT: Practical use case
3. PARAMETERS: Explain each parameter in plain language
4. REQUEST BODY: Explain required and optional fields (if applicable)
5. RESPONSES: What each status code means
6. COMMON MISTAKES: 1-2 pitfalls to avoid

Keep it concise and practical."""),
    ("human", "{endpoint_json}")
])

explain_chain = EXPLAIN_PROMPT | llm | StrOutputParser()

explanations = {}
for ep in ENDPOINTS:
    key = f"{ep['method']} {ep['path']}"
    explanation = explain_chain.invoke({"endpoint_json": json.dumps(ep, indent=2)})
    explanations[key] = explanation
    print(f"\n{key}")
    print("-" * 40)
    print(explanation[:400])
    if len(explanation) > 400:
        print(f"... ({len(explanation)} chars)")
    print()

## Step 6 — Generate Code Examples (Python, JavaScript, cURL)

For each endpoint we generate working code examples in three languages.
This is the most practical part of API documentation for developers.

In [ ]:
EXAMPLE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Generate a working code example for this API endpoint.
Use the specified language/tool. Include:
- Proper headers (Authorization, Content-Type)
- All required parameters with realistic sample values
- Error handling
- Brief inline comments

Base URL: {base_url}
Return ONLY the code — no explanation text, no markdown fences."""),
    ("human", "Endpoint: {method} {path}\nDetails: {details}\nLanguage: {language}")
])

example_chain = EXAMPLE_PROMPT | llm | StrOutputParser()

LANGUAGES = ["Python (requests)", "JavaScript (fetch)", "cURL"]
base_url = API_SPEC["servers"][0]["url"]

# Generate examples for the first 3 endpoints
code_examples = {}
for ep in ENDPOINTS[:3]:
    key = f"{ep['method']} {ep['path']}"
    code_examples[key] = {}
    print(f"\n{'=' * 60}")
    print(f"{key}: {ep['summary']}")

    for lang in LANGUAGES:
        example = example_chain.invoke({
            "base_url": base_url,
            "method": ep["method"],
            "path": ep["path"],
            "details": json.dumps(ep, indent=2)[:800],
            "language": lang,
        })
        code_examples[key][lang] = example
        print(f"\n  [{lang}]")
        print(f"  {example.strip()[:250]}")
        if len(example) > 250:
            print(f"  ... ({len(example)} chars)")

## Step 7 — Generate Quickstart Guide

We generate a **"Getting Started in 5 Minutes"** guide that walks
a developer through their first API interaction.

In [ ]:
QUICKSTART_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Write a "Getting Started in 5 Minutes" guide for this API.

Include these sections:
1. Prerequisites (API key, base URL)
2. Your first request — list tasks
3. Create a task
4. Update a task's status to done
5. Common patterns and tips

Use Python (requests) for all examples. Include realistic sample data.
Return valid Markdown."""),
    ("human", "API: {title} ({base_url})\nEndpoints:\n{endpoints}")
])

quickstart_chain = QUICKSTART_PROMPT | llm | StrOutputParser()

endpoints_summary = "\n".join(
    f"  {ep['method']} {ep['path']} — {ep['summary']}"
    for ep in ENDPOINTS
)

quickstart = quickstart_chain.invoke({
    "title": API_SPEC["info"]["title"],
    "base_url": base_url,
    "endpoints": endpoints_summary,
})

print("QUICKSTART GUIDE")
print("=" * 60)
print(quickstart[:2000])
if len(quickstart) > 2000:
    print(f"\n... ({len(quickstart)} chars total)")

## Step 8 — Generate Error Reference

We collect all response codes from the spec and generate an
**error reference table** explaining what each code means and
how to handle it.

In [ ]:
ERROR_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """Generate an error handling reference for this API.
For each HTTP status code, explain:
1. What it means
2. Common causes
3. How to fix it

Format as a Markdown table. Return ONLY Markdown."""),
    ("human", "API response codes:\n{codes}")
])

# Collect all unique response codes
all_codes = set()
for ep in ENDPOINTS:
    for r in ep["responses"]:
        all_codes.add((r["code"], r["description"]))

codes_text = "\n".join(f"  {code}: {desc}" for code, desc in sorted(all_codes))

error_chain = ERROR_PROMPT | llm | StrOutputParser()
error_ref = error_chain.invoke({"codes": codes_text})

print("ERROR REFERENCE")
print("=" * 50)
print(error_ref)

## Step 9 — Documentation Completeness Check

We verify that every endpoint from the spec has a corresponding
explanation in the generated documentation.

In [ ]:
print("DOCUMENTATION COMPLETENESS CHECK")
print("=" * 60)

print(f"\nEndpoint explanations:")
for ep in ENDPOINTS:
    key = f"{ep['method']} {ep['path']}"
    has_explanation = key in explanations
    status = "✓" if has_explanation else "✗ MISSING"
    print(f"  {status} {key}")

print(f"\nCode examples (first 3 endpoints):")
for ep in ENDPOINTS[:3]:
    key = f"{ep['method']} {ep['path']}"
    for lang in LANGUAGES:
        has_example = key in code_examples and lang in code_examples[key]
        status = "✓" if has_example else "✗"
        print(f"  {status} {key} [{lang}]")

print(f"\nGenerated artifacts:")
artifacts = [
    ("Endpoint explanations", len(explanations)),
    ("Code examples", sum(len(v) for v in code_examples.values())),
    ("Quickstart guide", 1 if quickstart else 0),
    ("Error reference", 1 if error_ref else 0),
]
for name, count in artifacts:
    print(f"  ✓ {name}: {count}")

## Step 10 — Full Documentation Summary

In [ ]:
print("API DOCUMENTATION SUMMARY")
print("=" * 60)
print(f"\nAPI: {API_SPEC['info']['title']} v{API_SPEC['info']['version']}")
print(f"Total endpoints: {len(ENDPOINTS)}")
print()

total_chars = 0
items = [
    ("Explanations", sum(len(v) for v in explanations.values())),
    ("Code examples", sum(len(e) for exs in code_examples.values() for e in exs.values())),
    ("Quickstart guide", len(quickstart)),
    ("Error reference", len(error_ref)),
]

print(f"{'Artifact':<25} {'Size':>8}")
print("-" * 35)
for name, size in items:
    total_chars += size
    print(f"{name:<25} {size:>6} chars")
print("-" * 35)
print(f"{'TOTAL':<25} {total_chars:>6} chars")
print(f"\nSpec size: {len(json.dumps(API_SPEC))} chars")
print(f"Doc-to-spec ratio: {total_chars / len(json.dumps(API_SPEC)):.1f}x")

## Evaluation Summary

| Dimension | How We Evaluated |
|---|---|
| **Spec parsing** | Verified `parse_openapi_spec()` extracts all 5 endpoints with params |
| **Explanation quality** | Checked each explanation covers what/when/params/responses |
| **Code examples** | Generated examples in 3 languages for first 3 endpoints |
| **Quickstart quality** | Verified guide includes setup + first request + create + update |
| **Error reference** | All unique status codes documented |
| **Completeness** | Every endpoint has an explanation (Step 9) |

### Known Limitations

- **No runtime testing**: Generated code examples are not executed.
  They may have subtle errors in headers or URL construction.
- **Schema depth**: Nested `$ref` schemas in real OpenAPI specs are not
  resolved by the parser — it only handles inline schemas.
- **Authentication**: The spec doesn't define security schemes, so
  auth handling in examples is assumed (Bearer token).
- **Rate limits**: Not present in this spec. Real APIs would need
  rate-limit documentation.
- **Webhooks / async**: Only REST endpoints are supported.

## How to Improve This Project

1. **`$ref` resolution** — resolve `$ref` pointers in real OpenAPI specs
2. **YAML support** — parse YAML-format OpenAPI specs alongside JSON
3. **Interactive explorer** — build a Streamlit UI for browsing endpoints
4. **SDK generation** — generate full client library wrappers, not just examples
5. **Test generation** — produce pytest tests for each endpoint
6. **Diff documentation** — compare two spec versions and document changes

## What You Learned

- **OpenAPI parsing** — extracting structured endpoint data from API specs
- **Plain-English explanations** — generating beginner-friendly API docs
- **Multi-language code examples** — Python, JavaScript, and cURL
- **Quickstart generation** — producing a getting-started guide
- **Error reference** — documenting all response codes with handling tips
- **Completeness verification** — checking generated docs cover the full spec